# Final Project Sains Data
## Prediksi Harga Mobil — CRISP-DM Pipeline

| | |
|---|---|
| **Dataset** | Car_sales.xls (157 records, 16 atribut) |
| **Metode** | CRISP-DM |
| **Algoritma** | Linear Regression |
| **Target** | Price_in_thousands |

---

## 📌 FASE 1 — Business Understanding

### 1.1 Latar Belakang & Problem Statement

Sebuah perusahaan manufaktur otomotif merekrut seorang Data Scientist untuk membangun sistem yang mampu:
1. **Merekomendasikan spesifikasi mobil** yang sedang laku di pasar berdasarkan data historis penjualan.
2. **Memprediksi harga mobil** yang tepat berdasarkan spesifikasi teknis yang diinputkan.
3. **Menyediakan antarmuka web** agar end user (tim manajemen produksi) dapat melakukan prediksi harga secara mandiri melalui browser.

Saat ini perusahaan belum memiliki sistem berbasis data — keputusan produksi masih bergantung pada intuisi, yang mengakibatkan:
- Potensi *overpricing* atau *underpricing* produk baru
- Ketidaksesuaian spesifikasi dengan selera pasar
- Proses estimasi harga memakan waktu berhari-hari

### 1.2 Tujuan Bisnis

| # | Tujuan | Kriteria Sukses |
|---|--------|-----------------|
| 1 | Memprediksi harga mobil berdasarkan spesifikasi dengan error minimal | R² Score ≥ 0.80 |
| 2 | Mengidentifikasi spesifikasi mobil yang laku di pasar | Top-10 analisis penjualan |
| 3 | Model dapat diakses via web oleh end user | Flask API aktif & responsif |

### 1.3 Pertanyaan Analitik

1. Mobil dengan spesifikasi seperti apa yang paling banyak terjual?
2. Atribut teknis mana yang paling berpengaruh terhadap harga mobil?
3. Seberapa akurat model Linear Regression dapat memprediksi harga?

### 1.4 Success Metrics

| Metrik | Target |
|--------|--------|
| R² Score | ≥ 0.80 |
| RMSE | < $4,000 (< 4 dalam satuan ribuan USD) |
| Response Time API | < 1 detik |

---

## 📌 FASE 2 — Data Understanding

### Langkah 1 — Memanggil Library yang Diperlukan

In [ ]:
# ── Data Manipulation ──
import pandas as pd
import numpy as np

# ── Visualization ──
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine Learning ──
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# ── Model Export ──
import joblib

# ── Misc ──
import warnings
warnings.filterwarnings('ignore')

# Styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ Semua library berhasil dimuat.')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   sklearn : ', end='')
import sklearn; print(sklearn.__version__)

### Langkah 2 — Load Data

In [ ]:
# Upload file Car_sales.xls ke Google Colab terlebih dahulu
# kemudian jalankan cell ini

from google.colab import files
uploaded = files.upload()   # pilih Car_sales.xls

In [ ]:
# Load dataset
# File .xls ini ternyata berformat CSV (comma-separated) — dibaca langsung dengan pd.read_csv
df = pd.read_csv('Car_sales.xls', sep=',')

print(f'✅ Dataset berhasil dimuat!')
print(f'   Jumlah baris   : {df.shape[0]}')
print(f'   Jumlah kolom   : {df.shape[1]}')
print(f'   Nama kolom     : {df.columns.tolist()}')

### Langkah 3 — Melihat Data

In [ ]:
# 3a. Tampilkan 5 baris pertama
print('=== HEAD (5 baris pertama) ===')
df.head()

In [ ]:
# 3b. Informasi tipe data dan non-null count
print('=== INFO DATASET ===')
df.info()

In [ ]:
# 3c. Statistik deskriptif
print('=== STATISTIK DESKRIPTIF ===')
df.describe().round(2)

In [ ]:
# 3d. Rangkuman missing value per kolom
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print('=== MISSING VALUE ===')
print(missing_df.to_string())

# Visualisasi missing value
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing_df.index, missing_df['Missing (%)'], color='#e07b54')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('Missing Value (%)')
ax.set_title('Persentase Missing Value per Kolom', fontweight='bold')
ax.set_xlim(0, 35)
plt.tight_layout()
plt.show()

print()
print('📝 Analisis:')
print('  - __year_resale_value memiliki missing value terbanyak (36 baris / ~22.9%).')
print('  - Kolom target Price_in_thousands hanya 2 missing value → baris tersebut akan dihapus.')
print('  - Kolom numerik lain (Engine_size, Horsepower, dll) missing sangat sedikit → imputasi dengan median.')

### Langkah 6 — Explorasi Data (EDA)

Pada tahap ini kita mengeksplorasi dataset untuk memahami pola penjualan dan karakteristik mobil yang paling diminati pasar.

In [ ]:
# ── 6a. Top 10 mobil dengan jumlah penjualan terbanyak ──

top10 = df.nlargest(10, 'Sales_in_thousands').copy()
top10['Car'] = top10['Manufacturer'] + ' ' + top10['Model']

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_d', 10)
bars = ax.barh(top10['Car'][::-1], top10['Sales_in_thousands'][::-1], color=colors)

for bar, val in zip(bars, top10['Sales_in_thousands'][::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,.1f}K', va='center', fontsize=9)

ax.set_xlabel('Penjualan (ribuan unit)', fontsize=11)
ax.set_title('Top 10 Mobil dengan Penjualan Terbanyak', fontsize=14, fontweight='bold')
ax.set_xlim(0, 620)
plt.tight_layout()
plt.show()

print()
print('📝 Analisis Top 10 Penjualan:')
print(f'  1. Ford F-Series mendominasi pasar dengan penjualan {top10.iloc[0]["Sales_in_thousands"]:,.0f}K unit —')
print(f'     hampir 2x lipat peringkat ke-2, menunjukkan dominasi segmen pickup/truck.')
print(f'  2. Ford Explorer & Toyota Camry berada di posisi 2-3, menandakan SUV & sedan')
print(f'     mid-size adalah segmen paling kompetitif.')
print(f'  3. Dari 10 besar, Ford mendominasi dengan 5 model sekaligus (F-Series, Explorer,')
print(f'     Taurus, Ranger, Focus), mencerminkan portofolio produk yang sangat kuat.')
print(f'  4. Honda Accord & Civic masuk 10 besar, mengindikasikan segmen city car &')
print(f'     compact sedan memiliki permintaan pasar yang tinggi.')

In [ ]:
# ── 6b. Harga dari 10 mobil paling laris ──

fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette('Greens_d', 10)
bars = ax.barh(top10['Car'][::-1], top10['Price_in_thousands'][::-1], color=palette)

for bar, val in zip(bars, top10['Price_in_thousands'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'${val:.2f}K', va='center', fontsize=9)

ax.set_xlabel('Harga (ribuan USD)', fontsize=11)
ax.set_title('Harga 10 Mobil dengan Penjualan Terbanyak', fontsize=14, fontweight='bold')
ax.set_xlim(0, 42)
plt.tight_layout()
plt.show()

print()
print('📝 Analisis Harga Top 10:')
print(f'  - Rentang harga: ${top10["Price_in_thousands"].min():.2f}K – ${top10["Price_in_thousands"].max():.2f}K')
print(f'  - Rata-rata harga top 10: ${top10["Price_in_thousands"].mean():.2f}K')
print(f'  - Ford Explorer ($31.93K) adalah yang paling mahal di antara 10 besar,')
print(f'    didorong oleh segmen SUV yang memiliki margin lebih tinggi.')
print(f'  - Ford Ranger ($12.05K) & Honda Civic ($12.89K) adalah yang termurah,')
print(f'    namun tetap masuk top 10 karena volume penjualan yang sangat tinggi.')
print(f'  - Ini menunjukkan bahwa harga terjangkau ($12K–$20K) lebih mendorong volume,')
print(f'    sementara harga premium ($25K–$32K) tetap kompetitif di segmen SUV/pickup.')

In [ ]:
# ── 6c. Minimal 3 variabel lain dari Top 10 mobil terlaris ──

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Atribut Teknis — 10 Mobil Terlaris', fontsize=14, fontweight='bold', y=1.02)

# (i) Horsepower
ax1 = axes[0]
colors1 = sns.color_palette('Oranges_d', 10)
bars1 = ax1.barh(top10['Car'][::-1], top10['Horsepower'][::-1], color=colors1)
for b, v in zip(bars1, top10['Horsepower'][::-1]):
    ax1.text(b.get_width()+1, b.get_y()+b.get_height()/2, f'{v:.0f} hp', va='center', fontsize=8)
ax1.set_xlabel('Horsepower (hp)')
ax1.set_title('Tenaga Mesin (HP)')
ax1.set_xlim(0, 280)

# (ii) Engine Size
ax2 = axes[1]
colors2 = sns.color_palette('Purples_d', 10)
bars2 = ax2.barh(top10['Car'][::-1], top10['Engine_size'][::-1], color=colors2)
for b, v in zip(bars2, top10['Engine_size'][::-1]):
    ax2.text(b.get_width()+0.05, b.get_y()+b.get_height()/2, f'{v:.1f}L', va='center', fontsize=8)
ax2.set_xlabel('Engine Size (Liter)')
ax2.set_title('Ukuran Mesin (L)')
ax2.set_xlim(0, 7)

# (iii) Fuel Efficiency
ax3 = axes[2]
colors3 = sns.color_palette('Reds_d', 10)
bars3 = ax3.barh(top10['Car'][::-1], top10['Fuel_efficiency'][::-1], color=colors3)
for b, v in zip(bars3, top10['Fuel_efficiency'][::-1]):
    ax3.text(b.get_width()+0.2, b.get_y()+b.get_height()/2, f'{v:.0f} mpg', va='center', fontsize=8)
ax3.set_xlabel('Fuel Efficiency (mpg)')
ax3.set_title('Efisiensi Bahan Bakar (mpg)')
ax3.set_xlim(0, 40)

plt.tight_layout()
plt.show()

print()
print('📝 Analisis 3 Atribut Teknis Top 10:')
print()
print('  [Horsepower]')
print(f'  - Dodge Ram Pickup memiliki HP tertinggi (230 hp), diikuti Ford F-Series (220 hp).')
print(f'  - Mobil dengan HP tinggi (>200) umumnya masuk segmen truck/pickup & SUV.')
print(f'  - Honda Civic memiliki HP terendah (106 hp), konsisten dengan posisinya sebagai city car efisien.')
print()
print('  [Engine Size]')
print(f'  - Dodge Ram Pickup & Ford F-Series menggunakan mesin terbesar (5.2L & 4.6L).')
print(f'  - Honda Civic & Ford Focus menggunakan mesin terkecil (1.6L & 2.0L) — cocok untuk efisiensi kota.')
print(f'  - Ada korelasi positif kuat antara engine size dan HP.')
print()
print('  [Fuel Efficiency]')
print(f'  - Honda Civic (32 mpg) & Ford Focus (30 mpg) adalah yang paling hemat bahan bakar.')
print(f'  - Dodge Ram Pickup (17 mpg) & Ford F-Series (18 mpg) paling boros — konsisten dengan mesin besar.')
print(f'  - Mobil dengan penjualan tertinggi cenderung berada di dua kutub: sangat hemat (city car)')
print(f'    atau kuat bertenaga (truck/SUV), bukan di tengah-tengah.')

In [ ]:
# ── Bonus: Correlation Matrix — membantu menentukan variabel untuk modeling ──

num_cols = ['Price_in_thousands', 'Engine_size', 'Horsepower', 'Wheelbase',
            'Width', 'Length', 'Curb_weight', 'Fuel_capacity', 'Fuel_efficiency',
            'Power_perf_factor', 'Sales_in_thousands']

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Semua Variabel Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Korelasi terhadap Price
price_corr = corr_matrix['Price_in_thousands'].drop('Price_in_thousands').sort_values(ascending=False)
print('📊 Korelasi terhadap Price_in_thousands (urut dari tertinggi):')
for col, val in price_corr.items():
    bar = '█' * int(abs(val) * 20)
    sign = '+' if val > 0 else '-'
    print(f'  {col:<25} {sign}{abs(val):.3f}  {bar}')

print()
print('📝 Insight:')
print('  - Engine_size, Horsepower, Curb_weight, dan Power_perf_factor')
print('    memiliki korelasi POSITIF KUAT dengan Price → semakin besar/bertenaga, semakin mahal.')
print('  - Fuel_efficiency berkorelasi NEGATIF dengan Price → mobil hemat cenderung lebih murah.')
print('  - Variabel-variabel ini akan menjadi kandidat utama sebagai fitur model prediksi.')

---
## ✅ Ringkasan Fase 1 & 2

| Fase | Status | Temuan Kunci |
|------|--------|--------------|
| Business Understanding | ✅ | Tujuan bisnis & success metrics terdefinisi (R² ≥ 0.80, RMSE < $4K) |
| Data Loading | ✅ | 157 baris × 16 kolom berhasil dimuat |
| Data Inspection | ✅ | Tipe data, statistik deskriptif, missing value teridentifikasi |
| EDA | ✅ | Top-10 penjualan, harga, HP, engine size, fuel efficiency dianalisis |

### Langkah Selanjutnya (Fase 3 — Data Preparation):
- Menghapus baris dengan missing value pada kolom target (`Price_in_thousands`)
- Imputasi kolom numerik lain dengan nilai median
- Memilih variabel independen berdasarkan korelasi terhadap harga
- Split data: 80% training — 20% testing
